In [ ]:
%load_ext autoreload
%autoreload 2 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
import os
import sys
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 500)
import importlib
import os
import sys

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)
from env.parameters import P
from analysis_functions.data_preparation import cohort_type_adjustment
import dask.dataframe as dd
import pickle
import yaml
from analysis_functions.feature_engineering import (
keep_england_country_imd,
drop_unknown_country_imd,
imd_quantiles,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
find_normal_boundaries, 
find_skewed_boundaries,
diagnostic_plots,
plot_boxplot_and_hist,
outlier_analysis,
keep_england_country_imd,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
impute_nulls_mice,
create_age_bands
)

from analysis_functions.custom_transformers import (
Custom_Winsoriser,
bmi_categoriser,
fev1fvc_ratio_categoriser,
traffic_intensity_quantiles,
inverse_distance_quantiles,
CustomFrequencyBinner,
CustomWaistBinner,
MultiTransform,
MultiTransformList,
CustomBMICategoriser,
CustomFev1FvcRatioCategoriser,
CustomInverseDistanceCategoriser,
CustomTrafficIntensityCategoriser,
ColumnSelector,
CustomBinaryCategoriserAroundMean,
CustomBinaryCategoriserAroundMedian,
CustomBinaryCategoriserAroundDecile
)
from scipy.stats.mstats import winsorize
from fancyimpute import IterativeImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer
from scipy.stats import shapiro
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
import warnings
from diffprivlib.utils import PrivacyLeakWarning
from sklearn.decomposition import PCA
import diffprivlib as dp
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
accuracy_score, confusion_matrix, 
classification_report, f1_score, 
roc_curve, roc_auc_score,
precision_recall_curve,
precision_score, recall_score, average_precision_score,balanced_accuracy_score, matthews_corrcoef)
import shap
from scipy.stats import chi2
from collections import Counter

from pipeline_functions import *
from custom_plots import *
from epi_functions import *

from sklearn.neighbors import NearestNeighbors
from scipy.stats import fisher_exact, norm, chi2_contingency
from statsmodels.stats.contingency_tables import Table2x2

In [ ]:
cohort_path = P.cohorts_ukb_start_gphesonly_path

In [ ]:
# Processed cohort
df_in = pd.read_csv(f'''{P.cohorts_ukb_start_gphesonly_path}/analysis_csv/epi_analysis_ready_df_ukb_start_gphesonly.csv''')

In [ ]:
# Column types
pickle_file = f'''{P.cohorts_ukb_start_gphesonly_path}/pickle/cols_dict_gphesonly_all.pickle'''

with open(pickle_file, 'rb') as f:
     cols_dict = pickle.load(f)

print(cols_dict.keys())

In [ ]:
# step 1. Adjust feature types
df_in = cohort_type_adjustment(df_in, cols_dict)


In [ ]:
# Countries
df_in["country_imd"].value_counts(dropna=False)

In [ ]:
df_in.shape

# How many records with less than 1 year of follow up?

In [ ]:
# Outcome feature
col_o = "flag_post_cohort_start_exac_ocs_y1"

In [ ]:
df_in[df_in["follow_up_asthma_pre_cohort_start"]<1].shape[0]

In [ ]:
df_in[df_in["follow_up_asthma_pre_cohort_start"]<1]["follow_up_asthma_pre_cohort_start"].describe()

In [ ]:
# Drop these
df = df_in[df_in["follow_up_asthma_pre_cohort_start"]>=1]

In [ ]:
df["follow_up_asthma_pre_cohort_start"].describe()

In [ ]:
df["follow_up_asthma_pre_cohort_start"].hist()

In [ ]:
df.loc[:, "follow_up_asthma_over_12"] = df["follow_up_asthma_pre_cohort_start"].apply(lambda x: 1 if x >=12 else 0)

In [ ]:
df["follow_up_asthma_over_12"].value_counts()

In [ ]:
df["age_asthma"].hist()

In [ ]:
df.loc[:, "late_onset_asthma_40"] = df["age_asthma"].apply(lambda x: 1 if x >=40 else 0)

In [ ]:
df["late_onset_asthma_40"].value_counts()

In [ ]:
# Any outcome on study start?

df[df['evdt_first_post_cohort_start_exac']==df['evdt_cohort_start']]

In [ ]:
print(df_in.shape)
print(df.shape)

In [ ]:
df_in.shape[0] - df.shape[0]

In [ ]:
df[col_o].value_counts()/df.shape[0]*100

In [ ]:
df[df['bmi_field']==30][['bmi_field', 'bmi_field_imputed', 'bmi_30_original']]

# Cardinal symptoms

In [ ]:
df[["wheeze_field", "shortness_breath_field", "chest_pain_field"]].head()

In [ ]:
df.loc[:, "cardinal_symptoms"] =df[["wheeze_field", "shortness_breath_field", "chest_pain_field"]].any(axis=1).astype(int)

In [ ]:
df["cardinal_symptoms"].value_counts()

# How many deaths before the outcome (do not censor). Just do sensitivity without these later

In [ ]:
df[["evdt_cohort_start","evdt_first_post_cohort_start_exac", "dod"]].dtypes

In [ ]:
condition = (df['dod'].notnull()) & (df['dod'] > df['evdt_cohort_start']) & (df['dod'] <= df['evdt_cohort_start'] + pd.Timedelta(days=365))

df[condition].shape

In [ ]:
# Percentage
df[condition].shape[0]/df.shape[0]*100

In [ ]:
# any exac in these?
df[condition & df[col_o]==1][["evdt_cohort_start", "evdt_first_post_cohort_start_exac", "dod"]].shape

In [ ]:
df[df[col_o]==1].shape[0]

In [ ]:
df[df[col_o]==0].shape[0]

In [ ]:
df[condition & df[col_o]==1][["evdt_cohort_start", "evdt_first_post_cohort_start_exac", "dod"]].head()

In [ ]:
df['tte_cohort_start_to_exac'].describe()

In [ ]:
# any deaths recorded before exac 1 event data? any deaths in non-exac within a year?

df[df['dod']<df['evdt_first_post_cohort_start_exac']]

In [ ]:
df[df['tte_cohort_start_to_exac']<0.003][['evdt_first_post_cohort_start_exac','evdt_cohort_start']]

# OHE Smoking and Ethnicity

- We have not dropped any levels here. Drop during modeling

In [ ]:
df = pd.get_dummies(df, columns=['eth_grouped_1b', 'desc_smoking_at_baseline'], drop_first=False)
dummy_columns = [col for col in df.columns if 'eth_grouped_1b_' in col or 'desc_smoking_at_baseline_' in col]
df[dummy_columns] = df[dummy_columns].astype(int)

In [ ]:
df['Smoker_current'] = df["desc_smoking_at_baseline_Current"].apply(lambda x: 1 if x ==1 else 0)

# Adjusted Odds ratio

In [ ]:
# Baseline random seed
np_random_seed = 7
# Differential privacy random seed
dp_random_seed = 47

In [ ]:
# Covariates
cov_list= ['age_60+', 'sex_female', 
           'eth_non_white',
            'pheno_anxiety_pre_cohort_start',
           'bmi_30_imputed', 
           'pheno_ckd_pre_cohort_start',
           'pheno_copd_pre_cohort_start',
           'pheno_cvd_pre_cohort_start',
           'pheno_diabetes_pre_cohort_start',
           'pheno_ht_pre_cohort_start',
           'cardinal_symptoms',
            'flag_pre_cohort_start_exac_y1', 'flag_pre_cohort_start_meds_ocs_y1'] 



In [ ]:
# Covariate rename
rename_dict = {
 'age_60+': "Age≥60",
 'sex_female': "Female sex",
 'eth_non_white': "Non white",
 'pheno_anxiety_pre_cohort_start': "Anxiety",
 'bmi_30_imputed': "BMI≥30",
 'pheno_ckd_pre_cohort_start': "CKD",
 'pheno_copd_pre_cohort_start' :"COPD",
 'pheno_cvd_pre_cohort_start': "CVD",
 'pheno_diabetes_pre_cohort_start': "Diabetes",
 'pheno_ht_pre_cohort_start' : "Hypertension",
 'cardinal_symptoms': "Cardinal symptomps",
 'flag_pre_cohort_start_exac_y1':  "Pre exacerbation",
 'flag_pre_cohort_start_meds_ocs_y1': "Pre OCS"
}

In [ ]:
df = df.rename(columns=rename_dict)

In [ ]:
cov_list = rename_dict.values()

In [ ]:
cov_list

In [ ]:
df[cov_list].head()

In [ ]:
df_prevalence, df_proportion = heatmap_df_maker(df, col_o, cov_list)
# Plot heatmap for Prevalence
plot_heatmap(df_prevalence, col_o, 'Prevalence (%)', cmap='Blues')

In [ ]:

# Plot heatmap for Proportion
plot_heatmap(df_proportion, col_o, 'Proportion (%)', cmap='Oranges')

In [ ]:
df_prevalence

In [ ]:
# Prevalence statistics in cases and controls
import numpy as np
import scipy.stats as stats

df_prevalence['Prevalence in cases (%)'] = df_prevalence['Prevalence in cases (%)'].round(2)
df_prevalence['Prevalence in controls (%)'] = df_prevalence['Prevalence in controls (%)'].round(2)

# Calculate Prevalence Ratio (PR) and prevalence difference
df_prevalence['Prevalence Ratio (PR)'] = df_prevalence['Prevalence in cases (%)'] / df_prevalence['Prevalence in controls (%)']
df_prevalence['Prevalence Ratio (PR)'] = df_prevalence['Prevalence Ratio (PR)'].round(2)
df_prevalence['Prevalence difference'] = df_prevalence['Prevalence in cases (%)'] - df_prevalence['Prevalence in controls (%)']

# Add placeholders for CI and p-values
df_prevalence['CI Lower'] = np.nan  # Placeholder for lower CI
df_prevalence['CI Upper'] = np.nan  # Placeholder for upper CI
df_prevalence['p-value'] = np.nan   # Placeholder for p-value

# Dummy denominators for proportions 
# Total number of cases
n1 = (df[col_o] == 1).sum()  
# Total number of control
n2 = (df[col_o] == 0).sum()  
print(n1)
print(n2)
# Calculate CI and p-values
for index, row in df_prevalence.iterrows():
    try:
        # Extract prevalence ratios and proportions
        pr = row['Prevalence Ratio (PR)']
        log_pr = np.log(pr) if pr > 0 else 0  # Avoid log(0)
        
        # Standard error of log PR
        se_log_pr = np.sqrt((1 / n1) + (1 / n2))
        
        # Confidence intervals for log PR
        ci_log_pr_lower = log_pr - 1.96 * se_log_pr
        ci_log_pr_upper = log_pr + 1.96 * se_log_pr
        ci_lower = np.exp(ci_log_pr_lower) if se_log_pr > 0 else np.nan
        ci_upper = np.exp(ci_log_pr_upper) if se_log_pr > 0 else np.nan
        
        # Calculate z-score and p-value
        z_score = log_pr / se_log_pr if se_log_pr > 0 else 0
        p_value = 2 * (1 - stats.norm.cdf(abs(z_score))) if se_log_pr > 0 else np.nan
        
        df_prevalence.at[index, 'CI Lower'] = round(ci_lower, 2)
        df_prevalence.at[index, 'CI Upper'] = round(ci_upper, 2)
        df_prevalence.at[index, 'p-value'] = round(p_value, 4)
    except Exception as e:
        print(f"Error at index {index}: {e}")

In [ ]:
df_prevalence

In [ ]:
df_prevalence.to_csv('6_a_adjusted_0747_df_prevalence.csv', index=False)  

In [ ]:
for item in df_prevalence['Prevalence in cases (%)']:
    print(item)

In [ ]:
df[col_o].value_counts()

In [ ]:
# count and percentage in cases
for item in cov_list:
    n = df[(df[item]==1)&(df[col_o]==1)].shape[0]
    denom = df[df[col_o]==1].shape[0]
    print(f'''{n} ({round(n/denom*100, 2)})''') 

In [ ]:
# count and percentage in controls

for item in cov_list:
    n = df[(df[item]==1)&(df[col_o]==0)].shape[0]
    denom = df[df[col_o]==0].shape[0]
    print(f'''{n} ({round(n/denom*100, 2)})''') 

In [ ]:
df["Pre exacerbation"].value_counts()

In [ ]:
df["Pre exacerbation"].value_counts()/df.shape[0]*100

In [ ]:
print(df[df[col_o]==1]["Pre exacerbation"].value_counts())
print(df[df[col_o]==1]["Pre exacerbation"].value_counts()/df[df[col_o]==1].shape[0]*100)

In [ ]:
print(df[df[col_o]==0]["Pre exacerbation"].value_counts())
print(df[df[col_o]==0]["Pre exacerbation"].value_counts()/df[df[col_o]==0].shape[0]*100)

In [ ]:
df["Pre OCS"].value_counts()

In [ ]:
df["Pre OCS"].value_counts()/df.shape[0]*100

In [ ]:
print(df[df[col_o]==1]["Pre OCS"].value_counts())
print(df[df[col_o]==1]["Pre OCS"].value_counts()/df[df[col_o]==1].shape[0]*100)

In [ ]:
print(df[df[col_o]==0]["Pre OCS"].value_counts())
print(df[df[col_o]==0]["Pre OCS"].value_counts()/df[df[col_o]==0].shape[0]*100)

In [ ]:
# Separate outocme feature
X = df[cov_list]

y = df[[col_o]].values.flatten()


# Non-DP logistic regression

In [ ]:

np_lr_model = LogisticRegression(random_state=np_random_seed)
np_lr_model.fit(X, y)

In [ ]:
# Co-efficients and odds ratios
np_coeff_df = make_OR_from_LR_output(np_lr_model, X)
np_coeff_df

In [ ]:
np_results_df = calculate_rr_or_ci_pvalue(np_lr_model, X, y)
np_results_df

In [ ]:
np_results_df.to_csv('6_a_adjusted_0747_df_np_results.csv', index=False)  

In [ ]:
def plot_forest_log_scale_method_22(df, type="RR", figsize=(10, 6), log_scale=True, xtick_fontsize=10, show_p_value=True):
    """Forest plot of a single model in log scale with ORs/RRs, CIs, and p-values listed on the right."""
    sns.set(style="white")

    if type == "OR":
        ratio_text = 'or'
        ci_lower_text = 'or_ci_lower'
        ci_upper_text = 'or_ci_upper'
        p_value_text = 'or_p_value'
        title_text = "Odds"
    else:
        ratio_text = 'rr'
        ci_lower_text = 'rr_ci_lower'
        ci_upper_text = 'rr_ci_upper'
        p_value_text = 'rr_p_value'
        title_text = "Risk"

    fig, ax = plt.subplots(figsize=figsize)

    ax.scatter(df[ratio_text], df['Covariate'], color='none')
    ax.invert_yaxis()

    for i, row in df.iterrows():
        fmt = 's'
        mfc_marker = 'white' if row[p_value_text] >= 0.05 else 'black'

        ax.errorbar(row[ratio_text], row['Covariate'],
                    xerr=[[row[ratio_text] - row[ci_lower_text]], [row[ci_upper_text] - row[ratio_text]]],
                    fmt=fmt, mfc=mfc_marker, color='black' if row[p_value_text] < 0.05 else 'black', ecolor='gray', capsize=0, markersize=4)
    # Add a vertical line at 1 (null effect)
    ax.axvline(x=1, color='gray', linestyle=':', linewidth=1)
    # Set x-axis to logarithmic scale
    if log_scale:
        ax.set_xscale('log')
    ax.set_xlabel(f'Adjusted {title_text} Ratio {"(log scale)" if log_scale else ""}')
    ax.set_title(f'Forest Plot of {title_text} Ratios with 95% Confidence Intervals')
    ax.grid(True, linestyle='--', alpha=0.2)
    ax.tick_params(axis='y', which='major', labelsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(True)  
    ax.yaxis.set_ticks_position('none') 

    # Add a column with OR/RR, CI, and p-value text outside the plot area
    xlim = ax.get_xlim()
    x_text_position = xlim[1] * 1.0 
    for i, row in df.iterrows():
        ratio_value = f"{row[ratio_text]:.2f}"
        ci_text = f"({row[ci_lower_text]:.2f}, {row[ci_upper_text]:.2f})"
        if row[p_value_text] < 0.001:
            p_value_text_display = "p < 0.001"
        elif row[p_value_text] < 0.01:
            p_value_text_display = "p < 0.01"
        elif row[p_value_text] < 0.05:
            p_value_text_display = "p < 0.05"
        else:
            p_value_text_display = f"p= {row[p_value_text]:.2g}"
        y_value = row['Covariate']
        ax_text = f"{ratio_value} {ci_text} {p_value_text_display}"
        if not show_p_value:
            ax_text = f"{ratio_value} {ci_text}"
        
        ax.text(x_text_position, y_value, 
                ax_text, 
                ha='left', va='center', fontsize=9, color='black')
    ax.set_xlim(left=0, right=6)
    ax.xaxis.set_tick_params(labelsize=xtick_fontsize)
    ax.margins(y=0.1) 
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_forest_log_scale_method_22(np_results_df, type="OR", figsize=(5, 5), log_scale= False, show_p_value=False)

In [ ]:
# Hosmer-Lemeshow statistics
np_pred_probs = np_lr_model.predict_proba(X)[:, 1]
hl_stat, p_value = hosmer_lemeshow_test(y, np_pred_probs)

print(f"Hosmer-Lemeshow statistic: {hl_stat:.4f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
# Calibration plot
from sklearn.calibration import calibration_curve

probs_model = np_lr_model.predict_proba(X)[:, 1]

fraction_of_positives, mean_predicted_value = calibration_curve(y, probs_model, n_bins=20)
plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="Logistic regression")
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.title('Calibration plot')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import brier_score_loss

probs_model = np_lr_model.predict_proba(X)[:, 1]
# Brier score
brier_score = brier_score_loss(y, probs_model)
print(f'Brier score: {brier_score}')

In [ ]:
from sklearn.metrics import roc_auc_score

# ROC AUC
roc = roc_auc_score(y, probs_model)

print(f'ROC AUC: {roc}')

In [ ]:

fpr, tpr, thresholds = roc_curve(y, probs_model)  # False Positive Rate, True Positive Rate
auc_score = roc_auc_score(y, probs_model)

# 4. Plot ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'ROC curve (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  # Diagonal line for random guessing
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# SHAP, Non-private

In [ ]:
shap.initjs()


In [ ]:
df_temp = pd.DataFrame(X, columns=cov_list)
df_temp = df_temp.apply(pd.to_numeric, errors='coerce')
df_temp = df_temp.astype('float64')
explainer = shap.LinearExplainer(np_lr_model, df_temp)
shap_values = explainer.shap_values(df_temp)
shap_explainer = explainer(df_temp)


In [ ]:
shap.plots.beeswarm(shap_explainer, max_display=len(cov_list), log_scale=True,  show=False)
plt.gcf().suptitle('Non-private')
plt.savefig('adjusted_shap_np.tiff', format='tiff', dpi=300, bbox_inches='tight')
plt.show()

# Differrential privacy

## Data norm (Binary only)

- Check how rare is the all one combination

In [ ]:
def row_to_tuple(row):
    return tuple(row)
combination_counts = Counter(df[cov_list].apply(row_to_tuple, axis=1))
comb_df = pd.DataFrame(list(combination_counts.items()), columns=['Combination', 'Count'])
comb_df = comb_df.sort_values('Count', ascending=False)
total_rows = len(df)
comb_df['Percentage'] = comb_df['Count'] / total_rows * 100


In [ ]:
print("\nSummary:")
print(f"Total number of rows: {total_rows}")
print(f"Number of unique combinations: {len(comb_df)}")
print(f"Most common combination occurs {comb_df['Count'].max()} times ({comb_df['Percentage'].max():.2f}%)")
print(f"Least common combination occurs {comb_df['Count'].min()} times ({comb_df['Percentage'].min():.2f}%)")
rare_combinations = comb_df[comb_df['Percentage'] < 1]
print(f"\nNumber of rare combinations (less than 1%): {len(rare_combinations)}")


In [ ]:
len(cov_list)

In [ ]:
data_norm = np.sqrt(len(cov_list))
data_norm

# DP across all epsilons

In [ ]:
epsilons = [0.01, 0.025,0.05, 0.1, 0.25, 0.693, 1, 1.098, 2,5, 10]


In [ ]:
results_df_2, coeff_df_2,  model_dict= make_dp_or_rr_all_covariate_all_outputs(X, y, 
                                                                               epsilons=epsilons, 
                                                                               data_norm=data_norm, 
                                                                               random_state=dp_random_seed)

In [ ]:
results_df_2.head()

In [ ]:
results_df_2.to_csv('6_a_adjusted_0747_df_differentially_private_outcomes.csv', index=False)  # Set index=False to avoid saving the DataFrame's index

In [ ]:
coeff_df_2

In [ ]:
# AIC BIC
aic_bic_results = {
        'epsilon': [x for x in model_dict.get('epsilon')],
        'AIC': [],
        'BIC': []
    }
log_likelihood_np = calculate_log_likelihood(model_dict.get('model')[0], X, y)
aic_np, bic_np = aic_bic(log_likelihood_np, X.shape[1], X.shape[0])
    
aic_bic_results['AIC'].append(aic_np)
aic_bic_results['BIC'].append(bic_np)
for e in reversed(epsilons):
    index = next(i for i, item in enumerate(model_dict['epsilon']) if item == e)
    dp_lr_model_temp = model_dict['model'][index]
    log_likelihood_dp = calculate_log_likelihood(dp_lr_model_temp, X, y)
    aic_dp, bic_dp = aic_bic(log_likelihood_dp, X.shape[1], X.shape[0])
    aic_bic_results['AIC'].append(aic_dp)
    aic_bic_results['BIC'].append(bic_dp)

aic_bic_df = pd.DataFrame(aic_bic_results)

In [ ]:
aic_bic_df

In [ ]:
plot_aic_bic(aic_bic_df)

In [ ]:
#plot_aic_bic_log(aic_bic_df)

In [ ]:
model_dict.keys()

In [ ]:
for i, e in enumerate(model_dict.get("epsilon")):
    print(i,  e)

In [ ]:
for i, e in enumerate(model_dict.get("epsilon")[1:]):
    print(i,  e)

In [ ]:
bland_altman_plot_all_epsilons(model_dict, X)

In [ ]:
calibrated_plot_all_epsilons(model_dict, X, y, fig_size=(4, 4))

In [ ]:

roc_curve_all_epsilons(model_dict, X, y, fig_size=(4, 4))

In [ ]:
# ROC AUc
probs_model1 = model_dict.get("model")[0].predict_proba(X)[:, 1]  # Probabilities for the positive class
c_indices = []
c_index = roc_auc_score(y, probs_model1)
c_indices.append(c_index)
model_names = [f'None']

for i, e in enumerate(model_dict.get("epsilon")[1:]):
    probs_model2 = model_dict.get("model")[i].predict_proba(X)[:, 1]
    c_index = roc_auc_score(y, probs_model2)
    c_indices.append(c_index)
    model_names.append(f'{e}')

lolipop_all_epsilons(c_indices, model_names, x_label= "ROC AUC")

In [ ]:
# Brier score
probs_model1 = model_dict.get("model")[0].predict_proba(X)[:, 1]  # Probabilities for the positive class
brier_scores = []

brier_score = brier_score_loss(y, probs_model1)
brier_scores.append(brier_score)
model_names = [f'None']

for i, e in enumerate(model_dict.get("epsilon")[1:]):
    probs_model2 = model_dict.get("model")[i + 1].predict_proba(X)[:, 1]
    brier_score = brier_score_loss(y, probs_model2)
    brier_scores.append(brier_score)
    model_names.append(f'{e}')


lolipop_all_epsilons(brier_scores, model_names, x_label= "Brier score")

In [ ]:
# Log-Loss
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt

probs_model1 = model_dict.get("model")[0].predict_proba(X)[:, 1]  # Probabilities for the positive class
log_losses = []

log_loss_value = log_loss(y, probs_model1)
log_losses.append(log_loss_value)
model_names = [f'None']

for i, e in enumerate(model_dict.get("epsilon")[1:]):
    probs_model2 = model_dict.get("model")[i + 1].predict_proba(X)[:, 1]
    log_loss_value = log_loss(y, probs_model2)
    log_losses.append(log_loss_value)
    model_names.append(f'{e}')

lolipop_all_epsilons(log_losses, model_names, x_label= "Log-loss", max_x=np.ceil(np.max(log_losses))+1 , label_space=0.2)

In [ ]:
# Weighted Log-Loss
from sklearn.metrics import log_loss
import numpy as np
import matplotlib.pyplot as plt

def get_class_weights(y):
    n_positives = np.sum(y)
    n_negatives = len(y) - n_positives
    weight_positive = len(y) / (2 * n_positives)
    weight_negative = len(y) / (2 * n_negatives)
    
    return weight_positive, weight_negative
weight_positive, weight_negative = get_class_weights(y)
sample_weights = np.array([weight_positive if label == 1 else weight_negative for label in y])
probs_model1 = model_dict.get("model")[0].predict_proba(X)[:, 1]  # Probabilities for the positive class
log_losses = []
log_loss_value = log_loss(y, probs_model1, sample_weight=sample_weights)
log_losses.append(log_loss_value)
model_names = [f'None']
for i, e in enumerate(model_dict.get("epsilon")[1:]):
    probs_model2 = model_dict.get("model")[i + 1].predict_proba(X)[:, 1]
    log_loss_value = log_loss(y, probs_model2, sample_weight=sample_weights)
    log_losses.append(log_loss_value)
    model_names.append(f'{e}')


In [ ]:
lolipop_all_epsilons(log_losses, model_names, x_label= "Weighted log-Loss", max_x=np.ceil(np.max(log_losses))+1 , label_space=0.2)

In [ ]:
print(weight_positive)
print(weight_negative)

In [ ]:
# Calculate all performance metrics
pipe_accuracy = []
pipe_f1 = []
pipe_roc= [] 
pipe_precision = []
pipe_recall = []
pipe_ap = []
pipe_balanced_accuracy = []
pipe_mcc = []
pipe_brier = []
pipe_logloss = []
pipe_weighted_logloss = []
y_pred_full = model_dict.get("model")[0].predict_proba(X)[:, 1] 
roc_auc_score_full = roc_auc_score(y, y_pred_full)
ap_full = average_precision_score(y, y_pred_full)
brier_score_full = brier_score_loss(y, y_pred_full)
log_loss_full = log_loss(y, y_pred_full)
weighted_log_loss_full = log_loss(y, y_pred_full, sample_weight=sample_weights)
for i, e in enumerate(model_dict.get("epsilon")[1:]):
    y_pred = model_dict.get("model")[i].predict_proba(X)[:, 1] 
    pipe_roc.append(roc_auc_score(y, y_pred))
    pipe_ap.append(average_precision_score(y, y_pred))
    pipe_brier.append(brier_score_loss(y, y_pred))
    pipe_logloss.append(log_loss(y, y_pred))
    pipe_weighted_logloss.append(log_loss(y, y_pred, sample_weight=sample_weights))


In [ ]:
def plot_df_2_new(epsilons, pipe_metric, metric_full, title, y_label, ylim_low=0, ylim_high=1, log_scale=True, save_as_tiff = ""):
    plt.figure(figsize=(6,4))
    if log_scale:
        plt.semilogx(epsilons, pipe_metric, label="Differentially private", zorder=10, color='black', 
                 marker='.', markersize=7, linestyle='-')
    else:
        plt.plot(epsilons, pipe_metric, label="Differentially private", zorder=10, color='black', 
                 marker='.', markersize=7, linestyle='-')
        
    plt.plot(epsilons, np.ones_like(epsilons) * metric_full, dashes=[2,2], label=f'Baseline {y_label}: {round(metric_full, 2)}', 
             zorder=5, color='gray', linestyle='--')
    plt.title(title)
    plt.xlabel("Epsilon in log scale" if log_scale else "Epsilon")
    plt.ylabel(y_label)
    plt.ylim(ylim_low, ylim_high)
    plt.xlim(epsilons[0], epsilons[-1])
    plt.legend(loc=2)
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tick_params(axis='both', which='major', labelsize=10)
    plt.tick_params(axis='both', which='minor', labelsize=8)
    plt.tight_layout()
    if save_as_tiff != "":
        plt.savefig(f'''{save_as_tiff}.tiff''', format='tiff', dpi=300)

    plt.show()

In [ ]:
# ROC AUC
plot_df_2_new(epsilons[::-1], pipe_roc, roc_auc_score_full, title="", y_label="ROC AUC", save_as_tiff="adjusted_roc")

In [ ]:
# ROC AUC
plot_df_2_new(epsilons[::-1], pipe_ap, ap_full, title="", y_label="AP")

In [ ]:
plot_df_2_new(epsilons[::-1], pipe_brier, brier_score_full, 
              title="", y_label="Brier score",
             ylim_low=0, ylim_high=0.5, save_as_tiff="brier_adjusted"
             )


In [ ]:
plot_df_2_new(epsilons[::-1], pipe_logloss, log_loss_full, 
              title="", y_label="Log-loss",
             ylim_low=0, ylim_high=5)

In [ ]:
plot_df_2_new(epsilons[::-1], pipe_weighted_logloss, weighted_log_loss_full, 
              title="Differentially private wighted log-loss score", y_label="",
             ylim_low=0, ylim_high=5)

# 

# All shaps

In [ ]:
explainer = shap.LinearExplainer(model_dict.get("model")[0], df_temp)
shap_values = explainer.shap_values(df_temp)
shap_values= explainer(df_temp)
shap.plots.beeswarm(shap_values, max_display=len(cov_list), log_scale=True, show=False)
plt.gcf().suptitle('Non-private')
plt.show()

In [ ]:

# Loop through the differentially private models and calculate SHAP
for i, e in enumerate(model_dict.get("epsilon")[1:]):
    explainer = shap.LinearExplainer(model_dict.get("model")[i], df_temp)

    shap_values = explainer.shap_values(df_temp)

    shap_values= explainer(df_temp)
    shap.plots.beeswarm(shap_values, max_display=len(cov_list), log_scale=True, show=False)
    plt.gcf().suptitle(f'Differentially private, epsilon= {e}')

    plt.show()

# Non logarithmic

In [ ]:
explainer = shap.LinearExplainer(model_dict.get("model")[0], df_temp)

shap_values = explainer.shap_values(df_temp)

shap_values= explainer(df_temp)
shap.plots.beeswarm(shap_values, max_display=len(cov_list), log_scale=False, show=False)
plt.gcf().suptitle('Non-private')

plt.show()

In [ ]:

# Loop through the differentially private models and calculate the Brier score
for i, e in enumerate(model_dict.get("epsilon")[1:]):
    explainer = shap.LinearExplainer(model_dict.get("model")[i], df_temp)

    shap_values = explainer.shap_values(df_temp)

    shap_values= explainer(df_temp)
    shap.plots.beeswarm(shap_values, max_display=len(cov_list), log_scale=False, show=False)
    plt.gcf().suptitle(f'Differentially private, epsilon= {e}')

    plt.show()

# BMI30

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("bmi_30_imputed")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("bmi_30_imputed")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),

                               title = "",
                               covariate_name_xlabel= "BMI≥30", 
                               type="OR",
                               logscale=False,
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values= True, 
                               show_p_value=False, xlim_min=0, xlim_max=3,  text_x_position=3.1
                          )

# Age >60

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("age_60+")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("age_60+")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),

                               title = "",
                               covariate_name_xlabel= "Age≥60",
                               type="OR",
                               logscale=False,
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False,  xlim_min=-0.1, xlim_max=2.9, text_x_position=3
                          )

# Sex female

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("sex_female")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("sex_female")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
  
                               title = "",
                               covariate_name_xlabel= "Female sex",
                               type="OR",
                               logscale=False,
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, xlim_min=-0.1, xlim_max=2.9,  text_x_position=3
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("sex_female")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
  
                               title = "",
                               covariate_name_xlabel= "Female sex",
                               type="OR",
                               logscale=False,
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, xlim_min=-0.1, xlim_max=4,  text_x_position=4.1, save_as_tiff="dp_adjusted_sex"
                          )

# Anxiety

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("pheno_anxiety_pre_cohort_start")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_anxiety_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
    
                               title = "",
                               covariate_name_xlabel= "Anxiety",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False,  xlim_min=0.8, xlim_max=3.8, text_x_position=3.9
                          )

# CKD

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("pheno_ckd_pre_cohort_start")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_ckd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),

                               title = "",
                               covariate_name_xlabel= "CKD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, xlim_min=0.5, xlim_max=3.5,  text_x_position=3.6
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_ckd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),

                               title = "",
                               covariate_name_xlabel= "CKD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, xlim_min=0, xlim_max=4,  text_x_position=4.1, save_as_tiff="dp_adjusted_ckd"
                          )

# COPD

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("pheno_copd_pre_cohort_start")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_copd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5,3.5),

                               title = "",
                               covariate_name_xlabel= "COPD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, xlim_min=2, xlim_max=5,  text_x_position=5.1
                          )

# CVD

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("pheno_cvd_pre_cohort_start")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_cvd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
      
                               title = "",
                               covariate_name_xlabel= "CVD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_cvd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(4.9, 4),
 
                               title = "",
                               covariate_name_xlabel= "CVD",
                               type="OR",
                               logscale=True, 
                               implausible_high_value= 15,
                               implausibly_high_text = "∞ (>15)",
                               show_all_values=True,
                               show_p_value=False
                          )

# Diabetes

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("pheno_diabetes_pre_cohort_start")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_diabetes_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),

                               title = "",
                               covariate_name_xlabel= "Diabetes",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                               xlim_min=0.1, 
                               xlim_max=3.1, 
                               text_x_position=3.3
                          )

# HT

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("pheno_ht_pre_cohort_start")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_ht_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),

                               title = "",
                               covariate_name_xlabel= "Hypertension",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False,
                                                            xlim_min=0.9, 
                               xlim_max=3.9, 
                               text_x_position=4
                               
                          )

# Cardinal symptoms

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("cardinal_symptoms")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("cardinal_symptoms")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
   
                               title = "",
                               covariate_name_xlabel= "Cardinal symptoms",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False,
                                xlim_min=0.3, 
                               xlim_max=3.3, 
                               text_x_position=3.4
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("cardinal_symptoms")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
   
                               title = "",
                               covariate_name_xlabel= "Cardinal symptoms",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False,
                                xlim_min=0, 
                               xlim_max=4, 
                               text_x_position=4.1, save_as_tiff="dp_adjusted_cardinal"
                          )

# Eth Non white

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("eth_non_white")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("eth_non_white")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
 
                               title = "",
                               covariate_name_xlabel= "Non-white ethnicity",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False,
                               xlim_min=0.8, 
                               xlim_max=3.8, 
                               text_x_position=3.9
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("eth_non_white")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
 
                               title = "",
                               covariate_name_xlabel= "Non-white ethnicity",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False,
                               xlim_min=0, 
                               xlim_max=4, 
                               text_x_position=4.1, save_as_tiff="dp_adjusted_ethnicity"
                          )

# Pre Exac

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("flag_pre_cohort_start_exac_y1")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("flag_pre_cohort_start_exac_y1")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
     
                               title = "",
                               covariate_name_xlabel= "1-year exacerbation\nclinical",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False,
                               xlim_min=3, 
                               xlim_max=6, 
                               text_x_position=6.1
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("flag_pre_cohort_start_exac_y1")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
     
                               title = "",
                               covariate_name_xlabel= "1-year exacerbation\nclinical",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False,
                               xlim_min=-0.3, 
                               xlim_max=6, 
                               text_x_position=6.1
                          )

# Pre OCs 

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("flag_pre_cohort_start_meds_ocs_y1")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("flag_pre_cohort_start_meds_ocs_y1")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
    
                               title = "",
                               covariate_name_xlabel= "1-year OCS prescription",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False,
                                xlim_min=-0.1, 
                               xlim_max=2.9, 
                               text_x_position=3
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("flag_pre_cohort_start_meds_ocs_y1")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
    
                               title = "",
                               covariate_name_xlabel= "1-year OCS prescription",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False,
                                xlim_min=0, 
                               xlim_max=4, 
                               text_x_position=4.1, save_as_tiff="dp_adjusted_ocs"
                          )